In [1]:
import pandas as pd
import zipfile
from pathlib import Path


In [ ]:
raw_path = Path.cwd()

zip_files = list(raw_path.glob("*.zip"))

with zipfile.ZipFile(zip_files[0], "r") as zip_ref:
    zip_ref.extractall(raw_path)

print("Dataset extracted successfully.")

In [ ]:
list(raw_path.rglob("*.csv"))

In [4]:
ratings_file = list(raw_path.rglob("ratings.csv"))[0]
movies_file = list(raw_path.rglob("movies.csv"))[0]

ratings = pd.read_csv(ratings_file)
movies = pd.read_csv(movies_file)

print("Ratings shape:", ratings.shape)
print("Movies shape:", movies.shape)

Ratings shape: (100836, 4)
Movies shape: (9742, 3)


In [5]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [6]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


## Data Validation

The ingested MovieLens data is validated before preprocessing.
Validation checks include schema validation, missing-value checks,
rating-range validation, duplicate detection, and referential integrity.

In [7]:
required_rating_columns = {'userId', 'movieId', 'rating', 'timestamp'}
required_movie_columns = {'movieId', 'title', 'genres'}

assert required_rating_columns.issubset(ratings.columns), \
    "Ratings schema validation failed."

assert required_movie_columns.issubset(movies.columns), \
    "Movies schema validation failed."

print("Schema validation passed.")

Schema validation passed.


In [8]:
print("Missing values in ratings:\n", ratings.isnull().sum())

print("Missing values in movies:\n", movies.isnull().sum())

Missing values in ratings:
 userId       0
movieId      0
rating       0
timestamp    0
dtype: int64
Missing values in movies:
 movieId    0
title      0
genres     0
dtype: int64


In [9]:
invalid_ratings = ratings[
    ~ratings['rating'].between(0.5, 5.0)
]

print("Invalid ratings:", len(invalid_ratings))

Invalid ratings: 0


In [10]:
duplicate_ratings = ratings.duplicated(subset=['userId', 'movieId'], keep=False).sum()

print("Duplicate ratings:", duplicate_ratings)

Duplicate ratings: 0


In [11]:
unknown_movies = set(ratings['movieId']) - set(movies['movieId'])

print("Unknown movies in ratings:", len(unknown_movies))

Unknown movies in ratings: 0


**Data Preparation**

In [12]:
ratings_clean = ratings.copy()
movies_clean = movies.copy()

In [13]:
print("Ratings duplicates:", ratings_clean.duplicated().sum())
print("Movies duplicates:", movies_clean.duplicated().sum())

Ratings duplicates: 0
Movies duplicates: 0


In [14]:
ratings_clean = ratings_clean.drop_duplicates(subset=['userId', 'movieId'], keep='first')

movies.clean = movies_clean.drop_duplicates(subset=['movieId'], keep='first')

C:\Users\omkar.s.SYNERGY\AppData\Local\Temp\ipykernel_7436\3410793984.py:3: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  movies.clean = movies_clean.drop_duplicates(subset=['movieId'], keep='first')


In [15]:
ratings_clean['rating_datetime'] = pd.to_datetime(
    ratings_clean['timestamp'],
    unit='s'
)

In [16]:
ratings_clean.head(5)

,userId,movieId,rating,timestamp,rating_datetime
0,1,1,4.0,964982703,2000-07-30 18:45:03
1,1,3,4.0,964981247,2000-07-30 18:20:47
2,1,6,4.0,964982224,2000-07-30 18:37:04
3,1,47,5.0,964983815,2000-07-30 19:03:35
4,1,50,5.0,964982931,2000-07-30 18:48:51


In [17]:
movies_clean.shape

(9742, 3)

In [18]:
movies_clean.head(5)

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [19]:
recommendation_data = ratings_clean.merge(
    movies_clean,
    on='movieId',
    how='inner'
)

print("Transformed dataset shape:", recommendation_data.shape)

recommendation_data.sample(5)

Transformed dataset shape: (100836, 7)


,userId,movieId,rating,timestamp,rating_datetime,title,genres
693,6,302,3.0,845555436,1996-10-17 12:30:36,"Queen Margot (Reine Margot, La) (1994)",Drama|Romance
84183,537,53322,5.0,1424138269,2015-02-17 01:57:49,Ocean's Thirteen (2007),Crime|Thriller
36097,246,32587,4.5,1354124511,2012-11-28 17:41:51,Sin City (2005),Action|Crime|Film-Noir|Mystery|Thriller
42076,287,4993,4.0,1110229952,2005-03-07 21:12:32,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy
43014,288,6942,3.0,1090941535,2004-07-27 15:18:55,Love Actually (2003),Comedy|Drama|Romance


In [20]:
movie_stats = (
    recommendation_data
    .groupby(['movieId', 'title'])
    .agg(
        avg_movie_rating=('rating', 'mean'),
        movie_rating_count=('rating', 'count')
    )
    .reset_index()
)

movie_stats.head()

,movieId,title,avg_movie_rating,movie_rating_count
0,1,Toy Story (1995),3.920930,215
1,2,Jumanji (1995),3.431818,110
2,3,Grumpier Old Men (1995),3.259615,52
3,4,Waiting to Exhale (1995),2.357143,7
4,5,Father of the Bride Part II (1995),3.071429,49


In [21]:
user_stats = (
    recommendation_data
    .groupby('userId')
    .agg(
        avg_user_rating=('rating', 'mean'),
        user_rating_count=('rating', 'count')
    )
    .reset_index()
)

user_stats.head()

,userId,avg_user_rating,user_rating_count
0,1,4.366379,232
1,2,3.948276,29
2,3,2.435897,39
3,4,3.555556,216
4,5,3.636364,44


In [22]:
feature_data = recommendation_data.merge(
    movie_stats,
    on=['movieId', 'title'],
    how='left'
)

feature_data = feature_data.merge(
    user_stats,
    on='userId',
    how='left'
)

feature_data.head()

,userId,movieId,rating,timestamp,rating_datetime,title,genres,avg_movie_rating,movie_rating_count,avg_user_rating,user_rating_count
0,1,1,4.0,964982703,2000-07-30 18:45:03,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.920930,215,4.366379,232
1,1,3,4.0,964981247,2000-07-30 18:20:47,Grumpier Old Men (1995),Comedy|Romance,3.259615,52,4.366379,232
2,1,6,4.0,964982224,2000-07-30 18:37:04,Heat (1995),Action|Crime|Thriller,3.946078,102,4.366379,232
3,1,47,5.0,964983815,2000-07-30 19:03:35,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,3.975369,203,4.366379,232
4,1,50,5.0,964982931,2000-07-30 18:48:51,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,4.237745,204,4.366379,232


Features stored in local system

In [23]:
from pathlib import Path

feature_store_path = Path("Feature_store")

In [24]:
feature_store_path.mkdir(parents=True, exist_ok=True)

movie_stats.to_csv(
    feature_store_path / "movie_stats.csv",
    index=False
)

In [25]:
user_stats.to_csv(
    feature_store_path / "user_features.csv",
    index=False
)

In [26]:
feature_data.to_csv(
    feature_store_path / "recommendation_features.csv",
    index=False
)


In [27]:
print("Feature Store files:")

for file in feature_store_path.glob("*.csv"):
    print(file.name)

Feature Store files:
movie_stats.csv
recommendation_features.csv
user_features.csv


Features Extracted for Model Training

In [28]:

feature_store_path = Path("Feature_store")

feature_data = pd.read_csv(
    feature_store_path / "recommendation_features.csv"
)

print(feature_data.shape)
feature_data.head()

(100836, 11)


,userId,movieId,rating,timestamp,rating_datetime,title,genres,avg_movie_rating,movie_rating_count,avg_user_rating,user_rating_count
0,1,1,4.0,964982703,2000-07-30 18:45:03,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.920930,215,4.366379,232
1,1,3,4.0,964981247,2000-07-30 18:20:47,Grumpier Old Men (1995),Comedy|Romance,3.259615,52,4.366379,232
2,1,6,4.0,964982224,2000-07-30 18:37:04,Heat (1995),Action|Crime|Thriller,3.946078,102,4.366379,232
3,1,47,5.0,964983815,2000-07-30 19:03:35,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,3.975369,203,4.366379,232
4,1,50,5.0,964982931,2000-07-30 18:48:51,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,4.237745,204,4.366379,232


In [29]:
user_movie_matrix = feature_data.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)

print("User-Movie Matrix Shape:", user_movie_matrix.shape)

user_movie_matrix.head()

User-Movie Matrix Shape: (610, 9724)


movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
user_movie_matrix_filled = user_movie_matrix.fillna(0)

Creating model by Score generating using Cosine similarity between user preferences.

In [36]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(user_movie_matrix_filled)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix_filled.index,
    columns=user_movie_matrix_filled.index
)

print("Similarity Matrix Shape:", user_similarity_df.shape)

user_similarity_df.head()


Similarity Matrix Shape: (610, 610)


userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
userId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.027283,0.059720,0.194395,0.129080,0.128152,0.158744,0.136968,0.064263,0.016875,...,0.080554,0.164455,0.221486,0.070669,0.153625,0.164191,0.269389,0.291097,0.093572,0.145321
2,0.027283,1.000000,0.000000,0.003726,0.016614,0.025333,0.027585,0.027257,0.000000,0.067445,...,0.202671,0.016866,0.011997,0.000000,0.000000,0.028429,0.012948,0.046211,0.027565,0.102427
3,0.059720,0.000000,1.000000,0.002251,0.005020,0.003936,0.000000,0.004941,0.000000,0.000000,...,0.005048,0.004892,0.024992,0.000000,0.010694,0.012993,0.019247,0.021128,0.000000,0.032119
4,0.194395,0.003726,0.002251,1.000000,0.128659,0.088491,0.115120,0.062969,0.011361,0.031163,...,0.085938,0.128273,0.307973,0.052985,0.084584,0.200395,0.131746,0.149858,0.032198,0.107683
5,0.129080,0.016614,0.005020,0.128659,1.000000,0.300349,0.108342,0.429075,0.000000,0.030611,...,0.068048,0.418747,0.110148,0.258773,0.148758,0.106435,0.152866,0.135535,0.261232,0.060792


In [37]:
model_path = Path("models")
model_path.mkdir(parents=True, exist_ok=True)

user_similarity_df.to_pickle(
    model_path / "user_similarity_model.pkl"
)

print("Recommendation model saved successfully.")

Recommendation model saved successfully.


In [38]:
def recommend_movies(
    user_id,
    user_movie_matrix,
    user_similarity_df,
    movies,
    n_recommendations=5,
    n_similar_users=20
):
    # Check whether user exists
    if user_id not in user_movie_matrix.index:
        return f"User {user_id} does not exist."

    # Movies already rated by target user
    user_ratings = user_movie_matrix.loc[user_id]

    already_rated = set(
        user_ratings[user_ratings.notna()].index
    )

    # Find most similar users
    similar_users = (
        user_similarity_df[user_id]
        .drop(user_id)
        .sort_values(ascending=False)
        .head(n_similar_users)
    )

    # Candidate movie scores
    weighted_scores = {}
    similarity_totals = {}

    for similar_user_id, similarity in similar_users.items():

        ratings_from_similar_user = user_movie_matrix.loc[similar_user_id]

        for movie_id, rating in ratings_from_similar_user.dropna().items():

            # Do not recommend something already rated
            if movie_id in already_rated:
                continue

            weighted_scores[movie_id] = (
                weighted_scores.get(movie_id, 0)
                + rating * similarity
            )

            similarity_totals[movie_id] = (
                similarity_totals.get(movie_id, 0)
                + similarity
            )

    # Calculate recommendation score
    recommendation_scores = {
        movie_id: weighted_scores[movie_id] / similarity_totals[movie_id]
        for movie_id in weighted_scores
        if similarity_totals[movie_id] > 0
    }

    # Select highest-scoring movies
    top_movies = sorted(
        recommendation_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )[:n_recommendations]

    recommendations = pd.DataFrame(
        top_movies,
        columns=["movieId", "predicted_rating"]
    )

    # Add movie details
    recommendations = recommendations.merge(
        movies[["movieId", "title", "genres"]],
        on="movieId",
        how="left"
    )

    return recommendations[
        ["movieId", "title", "genres", "predicted_rating"]
    ]

Testing of model

In [47]:
user_id = 45

recommendations = recommend_movies(
    user_id=user_id,
    user_movie_matrix=user_movie_matrix,
    user_similarity_df=user_similarity_df,
    movies=movies_clean,
    n_recommendations=5
)

recommendations

,movieId,title,genres,predicted_rating
0,7387,Dawn of the Dead (1978),Action|Drama|Horror,5.0
1,4235,Amores Perros (Love's a Bitch) (2000),Drama|Thriller,5.0
2,1203,12 Angry Men (1957),Drama,5.0
3,1665,Bean (1997),Comedy,5.0
4,2582,Twin Dragons (Shuang long hui) (1992),Action|Comedy,5.0


In [44]:
rated_by_user = set(
    user_movie_matrix.loc[user_id]
    .dropna()
    .index
)

recommended_movies = set(
    recommendations["movieId"]
)

print(
    "Already-rated movies in recommendations:",
    rated_by_user.intersection(recommended_movies)
)

Already-rated movies in recommendations: set()
